In [1]:
import os
%pwd
os.chdir("../")
%pwd

'/home/harris/pers/Kidney-classification'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    """Base configuration for preparing models."""
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [3]:
from CNNClassifier.constants import *
from CNNClassifier.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        create_directories([config.root_dir]) 

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES,
            params_learning_rate=self.params.LEARNING_RATE,
        )

        return prepare_base_model_config

In [5]:
import os
import urllib.request as request
import zipfile as zipfile
import tensorflow as tf
from CNNClassifier import logger


2025-04-30 16:28:34.634516: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-30 16:28:34.640356: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-30 16:28:34.660178: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746012514.689861   13018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746012514.698895   13018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746012514.719497   13018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            include_top=self.config.params_include_top,
            weights=self.config.params_weights,
            input_shape=self.config.params_image_size,
        )
        self.save_model(path = self.config.base_model_path, model = self.model)
        logger.info(f"Base model downloaded and saved at {self.config.base_model_path}")
        
    
    @staticmethod
    def _prepare_full_model(model, classes,freeze_all,freeze_till,learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes, activation="softmax")(flatten_in)
        
        full_model = tf.keras.models.Model(inputs=model.input, outputs=prediction)
        
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"],
            run_eagerly=True,
        )
        full_model.summary()
        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate,
        )
        self.save_model(path = self.config.updated_base_model_path, model = self.full_model)
        logger.info(f"Updated model saved at {self.config.updated_base_model_path}")
                
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
        logger.info(f"Model saved at {path}")
        
    
        

In [7]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
    
except Exception as e:
    logger.exception(e)
    raise e

[2025-04-30 16:28:39,311: INFO: common]: YAML file config/config.yaml loaded successfully.
[2025-04-30 16:28:39,318: INFO: common]: YAML file params.yaml loaded successfully.
[2025-04-30 16:28:39,321: INFO: common]: Directory created successfully at: artifacts
[2025-04-30 16:28:39,324: INFO: common]: Directory created successfully at: artifacts/prepare_base_model


E0000 00:00:1746012519.371939   13018 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1746012519.373922   13018 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


[2025-04-30 16:28:40,240: WARNING: saving_api]: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
[2025-04-30 16:28:48,657: INFO: 3166666553]: Model saved at artifacts/prepare_base_model/base_model.h5
[2025-04-30 16:28:48,657: INFO: 3166666553]: Base model downloaded and saved at artifacts/prepare_base_model/base_model.h5
[2025-04-30 16:28:48,658: ERROR: 85202762]: PrepareBaseModel._prepare_full_model() got an unexpected keyword argument 'run_eagerly'
Traceback (most recent call last):
  File "/tmp/ipykernel_13018/85202762.py", line 6, in <module>
    prepare_base_model.update_base_model()
  File "/tmp/ipykernel_13018/3166666553.py", line 39, in update_base_model
    self.full_model = self._prepare_full_model(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^
T

TypeError: PrepareBaseModel._prepare_full_model() got an unexpected keyword argument 'run_eagerly'